In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('train.csv.zip')
df.dropna(inplace=True)

In [4]:
new_df = df.sample(100000, random_state=2)

In [5]:
new_df.shape

(100000, 6)

In [6]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 331535 to 166469
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            100000 non-null  int64 
 1   qid1          100000 non-null  int64 
 2   qid2          100000 non-null  int64 
 3   question1     100000 non-null  object
 4   question2     100000 non-null  object
 5   is_duplicate  100000 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 5.3+ MB


In [ ]:
print(new_df['is_duplicate'].value_counts())

is_duplicate
0    62996
1    37004
Name: count, dtype: int64


In [ ]:
ques_df = new_df[['question1', 'question2']]

#feature engineering

def common_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split()))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split()))
    return len(w1 & w2)

def total_words(row):
    w1 = set(map(lambda word: word.lower().strip(), row['question1'].split()))
    w2 = set(map(lambda word: word.lower().strip(), row['question2'].split()))
    return (len(w1) + len(w2))

new_df['q1_len'] = new_df['question1'].str.len()
new_df['q2_len'] = new_df['question2'].str.len()
new_df['q1_num_words'] = new_df['question1'].apply(lambda row: len(row.split()))
new_df['q2_num_words'] = new_df['question2'].apply(lambda row: len(row.split()))
new_df['word_common'] = new_df.apply(common_words, axis=1)
new_df['word_total'] = new_df.apply(total_words, axis=1)
new_df['word_share'] = round(new_df['word_common']/new_df['word_total'],2)

new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,q2_num_words,word_common,word_total,word_share
331535,331535,169053,295926,How can I learn Norwegian?,What is the quickest way to learn Norwegian?,1,26,44,5,8,2,13,0.15
45407,45407,81383,81384,How are currency rates determined?,Where and how are exchange rates determined?,1,34,44,5,7,4,12,0.33
286200,286200,285024,406729,What is substitution?,What is a substitute for caciocavallo?,0,21,38,3,6,2,9,0.22
157195,157195,245856,245857,How can I make iPhone 4s faster with IOS 9.2?,I have an iPhone 4S. How do I make it faster a...,1,45,73,10,16,5,25,0.20
154346,154346,242075,242076,How can I help my girlfriend cope with her par...,What can I do to help my girlfriend through he...,1,59,64,11,12,8,23,0.35


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Vectorizing questions
cv = CountVectorizer(max_features=2000)
q1_bow = cv.fit_transform(new_df['question1']).toarray()
q2_bow = cv.transform(new_df['question2']).toarray()

In [ ]:
from scipy.sparse import hstack, csr_matrix
from scipy import sparse

temp_df = new_df.drop(columns=['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'])
X = sparse.hstack((q1_bow, q2_bow, csr_matrix(temp_df.values)), format='csr')
y = new_df['is_duplicate'].values

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7918


In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    device='cuda',
    n_estimators=500
)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("Dataset Accuracy (GPU):", accuracy_score(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:52:16] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:52:16] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


Dataset Accuracy (GPU): 0.79575


In [ ]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
STOP_WORDS = set(stopwords.words("english"))

def get_token_features(row):
    q1 = str(row['question1']).lower().split()
    q2 = str(row['question2']).lower().split()

    if len(q1) == 0 or len(q2) == 0:
        return [0.0] * 8

    # Non-stopwords
    q1_words = set([w for w in q1 if w not in STOP_WORDS])
    q2_words = set([w for w in q2 if w not in STOP_WORDS])

    # Stopwords
    q1_stops = set([w for w in q1 if w in STOP_WORDS])
    q2_stops = set([w for w in q2 if w in STOP_WORDS])

    common_word_count = len(q1_words.intersection(q2_words))
    common_stop_count = len(q1_stops.intersection(q2_stops))
    common_token_count = len(set(q1).intersection(set(q2)))

    # Ratios
    token_features = [
        common_word_count / (min(len(q1_words), len(q2_words)) + 1e-6),
        common_word_count / (max(len(q1_words), len(q2_words)) + 1e-6),
        common_stop_count / (min(len(q1_stops), len(q2_stops)) + 1e-6),
        common_stop_count / (max(len(q1_stops), len(q2_stops)) + 1e-6),
        common_token_count / (min(len(q1), len(q2)) + 1e-6),
        common_token_count / (max(len(q1), len(q2)) + 1e-6),
        int(q1[0] == q2[0]),
        int(q1[-1] == q2[-1])
    ]
    return token_features


token_features = new_df.apply(get_token_features, axis=1)
new_df[['cwc_min', 'cwc_max', 'csc_min', 'csc_max', 'ctc_min', 'ctc_max', 'last_word_eq', 'first_word_eq']] = pd.DataFrame(token_features.tolist(), index=new_df.index)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
new_df['abs_len_diff'] = abs(new_df['q1_len'] - new_df['q2_len'])
new_df['mean_len'] = (new_df['q1_len'] + new_df['q2_len']) / 2

# Word-based length features
new_df['abs_word_diff'] = abs(new_df['q1_num_words'] - new_df['q2_num_words'])
new_df['mean_word_len'] = (new_df['q1_num_words'] + new_df['q2_num_words']) / 2

In [ ]:
!pip install rapidfuzz
from rapidfuzz import fuzz

def get_fuzzy_features(row):
    q1 = str(row['question1'])
    q2 = str(row['question2'])

    return [
        fuzz.QRatio(q1, q2),
        fuzz.partial_ratio(q1, q2),
        fuzz.token_sort_ratio(q1, q2),
        fuzz.token_set_ratio(q1, q2)
    ]

fuzzy_features = new_df.apply(get_fuzzy_features, axis=1)
new_df[['fuzz_ratio', 'fuzz_partial_ratio', 'token_sort_ratio', 'token_set_ratio']] = pd.DataFrame(fuzzy_features.tolist(), index=new_df.index)

In [ ]:
temp_df.head()

,q1_len,q2_len,q1_num_words,q2_num_words,word_common,word_total,word_share,cwc_min,cwc_max,csc_min,...,last_word_eq,first_word_eq,abs_len_diff,mean_len,abs_word_diff,mean_word_len,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio
331535,26,44,5,8,2,13,0.15,1.000000,0.500000,0.000000,...,0,1,18,35.0,3,6.5,60.000000,81.818182,51.428571,76.190476
45407,34,44,5,7,4,12,0.33,0.666666,0.666666,1.000000,...,0,1,10,39.0,2,6.0,69.230769,80.597015,58.974359,76.363636
286200,21,38,3,6,2,9,0.22,0.000000,0.000000,1.000000,...,1,0,17,29.5,3,4.5,67.796610,85.000000,57.627119,57.627119
157195,45,73,10,16,5,25,0.20,0.500000,0.500000,0.500000,...,0,0,28,59.0,6,13.0,42.372881,51.111111,57.627119,69.565217
154346,59,64,11,12,8,23,0.35,1.000000,0.800000,0.666667,...,0,1,5,61.5,1,11.5,79.674797,81.481481,78.048780,86.538462


In [ ]:
from scipy.sparse import hstack, csr_matrix
from scipy import sparse

temp_df = new_df.drop(columns=['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'])
X = sparse.hstack((q1_bow, q2_bow, csr_matrix(temp_df.values)), format='csr')
y = new_df['is_duplicate'].values

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    tree_method='hist',
    device='cuda',
    n_estimators=500
)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:00:53] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:00:53] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


Dataset Accuracy (GPU): 0.79575


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.80615


#Finalized Method


In [8]:
!pip install rapidfuzz
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
from nltk.corpus import stopwords
from rapidfuzz import fuzz
import nltk
nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
df = pd.read_csv('train.csv.zip')
df.dropna(inplace=True)
new_df = df.sample(100000, random_state=2)
STOP_WORDS = set(stopwords.words("english"))

Feature Engineering


In [10]:
new_df['q1_len']       = new_df['question1'].str.len()
new_df['q2_len']       = new_df['question2'].str.len()
new_df['q1_num_words'] = new_df['question1'].apply(lambda x: len(x.split()))
new_df['q2_num_words'] = new_df['question2'].apply(lambda x: len(x.split()))
new_df['abs_len_diff'] = abs(new_df['q1_len'] - new_df['q2_len'])
new_df['mean_len']     = (new_df['q1_len'] + new_df['q2_len']) / 2
new_df['abs_word_diff']  = abs(new_df['q1_num_words'] - new_df['q2_num_words'])
new_df['mean_word_len']  = (new_df['q1_num_words'] + new_df['q2_num_words']) / 2

In [11]:
def common_words(row):
    w1 = set(row['question1'].lower().split())
    w2 = set(row['question2'].lower().split())
    return len(w1 & w2)

def total_words(row):
    w1 = set(row['question1'].lower().split())
    w2 = set(row['question2'].lower().split())
    return len(w1) + len(w2)

In [12]:
new_df['word_common'] = new_df.apply(common_words, axis=1)
new_df['word_total']  = new_df.apply(total_words, axis=1)
new_df['word_share']  = round(new_df['word_common'] / new_df['word_total'], 2)

In [13]:
def get_token_features(row):
    q1 = str(row['question1']).lower().split()
    q2 = str(row['question2']).lower().split()
    if not q1 or not q2:
        return [0.0] * 8
    q1_words  = set(w for w in q1 if w not in STOP_WORDS)
    q2_words  = set(w for w in q2 if w not in STOP_WORDS)
    q1_stops  = set(w for w in q1 if w in STOP_WORDS)
    q2_stops  = set(w for w in q2 if w in STOP_WORDS)
    cwc = len(q1_words & q2_words)
    csc = len(q1_stops & q2_stops)
    ctc = len(set(q1) & set(q2))
    return [
        cwc / (min(len(q1_words), len(q2_words)) + 1e-6),
        cwc / (max(len(q1_words), len(q2_words)) + 1e-6),
        csc / (min(len(q1_stops), len(q2_stops)) + 1e-6),
        csc / (max(len(q1_stops), len(q2_stops)) + 1e-6),
        ctc / (min(len(q1), len(q2)) + 1e-6),
        ctc / (max(len(q1), len(q2)) + 1e-6),
        int(q1[0] == q2[0]),
        int(q1[-1] == q2[-1])
    ]

In [14]:
token_features = new_df.apply(get_token_features, axis=1)
new_df[['cwc_min','cwc_max','csc_min','csc_max',
        'ctc_min','ctc_max','first_word_eq','last_word_eq']] = pd.DataFrame(token_features.tolist(), index=new_df.index)

In [15]:
def get_fuzzy_features(row):
    q1, q2 = str(row['question1']), str(row['question2'])
    return [
        fuzz.QRatio(q1, q2),
        fuzz.partial_ratio(q1, q2),
        fuzz.token_sort_ratio(q1, q2),
        fuzz.token_set_ratio(q1, q2)
    ]

fuzzy_features = new_df.apply(get_fuzzy_features, axis=1)
new_df[['fuzz_ratio','fuzz_partial_ratio',
        'token_sort_ratio','token_set_ratio']] = pd.DataFrame(fuzzy_features.tolist(), index=new_df.index)


Build Train-Test Split

In [16]:
feature_cols = [
    'q1_len','q2_len','q1_num_words','q2_num_words',
    'abs_len_diff','mean_len','abs_word_diff','mean_word_len',
    'word_common','word_total','word_share',
    'cwc_min','cwc_max','csc_min','csc_max',
    'ctc_min','ctc_max','first_word_eq','last_word_eq',
    'fuzz_ratio','fuzz_partial_ratio','token_sort_ratio','token_set_ratio'
]

X = new_df[feature_cols].values
y = new_df['is_duplicate'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [24]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.763


In [25]:
# XGBoost
neg, pos = np.bincount(y_train)
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg/pos,
    tree_method='hist', random_state=1, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
print("XGBoost Accuracy:", xgb_acc)

XGBoost Accuracy: 0.76545


BOW + Features

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

cv = CountVectorizer(max_features=3000)
q1_bow = cv.fit_transform(new_df['question1'])
q2_bow = cv.transform(new_df['question2'])

In [27]:
# Combine BoW + advanced features
X_hand = new_df[feature_cols].values
X_bow = sparse.hstack([q1_bow, q2_bow, sparse.csr_matrix(X_hand)], format='csr')
y = new_df['is_duplicate'].values

In [28]:
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=1
)

In [29]:
# Random Forest
rf2 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=1, n_jobs=-1)
rf2.fit(X_train_bow, y_train_bow)
rf2_acc = accuracy_score(y_test_bow, rf2.predict(X_test_bow))
print("Random Forest Accuracy:", rf2_acc)

Random Forest Accuracy: 0.8035


In [30]:
# XGBoost
neg, pos = np.bincount(y_train_bow.toarray().ravel() if hasattr(y_train_bow, 'toarray') else y_train_bow)
xgb2 = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg/pos,
    tree_method='hist', random_state=1, eval_metric='logloss'
)
xgb2.fit(X_train_bow, y_train_bow)
xgb2_acc = accuracy_score(y_test_bow, xgb2.predict(X_test_bow))
print("XGBoost Accuracy:", xgb2_acc)

XGBoost Accuracy: 0.79815


TF-IDF + Features

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
q1_tfidf = tfidf.fit_transform(new_df['question1'])
q2_tfidf = tfidf.transform(new_df['question2'])

X_tfidf = sparse.hstack([q1_tfidf, q2_tfidf, sparse.csr_matrix(X_hand)], format='csr')

X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=1
)

In [32]:
rf3 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=1, n_jobs=-1)
rf3.fit(X_train_tfidf, y_train_tfidf)
rf3_acc = accuracy_score(y_test_tfidf, rf3.predict(X_test_tfidf))
print("Random Forest Accuracy:", rf3_acc)

Random Forest Accuracy: 0.8033


In [33]:
neg3, pos3 = np.bincount(y_train_tfidf.toarray().ravel() if hasattr(y_train_tfidf, 'toarray') else y_train_tfidf)
xgb3 = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg3/pos3,
    tree_method='hist', random_state=1, eval_metric='logloss'
)
xgb3.fit(X_train_tfidf, y_train_tfidf)
xgb3_acc = accuracy_score(y_test_tfidf, xgb3.predict(X_test_tfidf))
print("XGBoost Accuracy:", xgb3_acc)

XGBoost Accuracy: 0.79425


Sentence Embeddings + Features

In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

In [18]:
encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

q1_emb = encoder.encode(new_df['question1'].tolist(), batch_size=256, show_progress_bar=True)
q2_emb = encoder.encode(new_df['question2'].tolist(), batch_size=256, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

In [19]:
cosine    = np.array([cos_sim([q1_emb[i]], [q2_emb[i]])[0][0] for i in range(len(q1_emb))])
diff      = np.abs(q1_emb - q2_emb)
product   = q1_emb * q2_emb

In [20]:
X_hand = new_df[feature_cols].values
X_emb = np.hstack([diff, product, cosine.reshape(-1, 1), X_hand])

In [21]:
X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_emb, y, test_size=0.2, random_state=1
)

In [22]:
# Random Forest
rf4 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=1, n_jobs=-1)
rf4.fit(X_train_emb, y_train_emb)
rf4_acc = accuracy_score(y_test_emb, rf4.predict(X_test_emb))
print("Random Forest Accuracy:", rf4_acc)

Random Forest Accuracy: 0.82815


In [23]:
# XGBoost
neg4, pos4 = np.bincount(y_train_emb if isinstance(y_train_emb, np.ndarray) else y_train_emb.toarray().ravel())
xgb4 = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg4/pos4,
    tree_method='hist', random_state=1, eval_metric='logloss'
)
xgb4.fit(X_train_emb, y_train_emb)
xgb4_acc = accuracy_score(y_test_emb, xgb4.predict(X_test_emb))
print("XGBoost Accuracy:", xgb4_acc)

XGBoost Accuracy: 0.86435
